# 01 -- Build dataset (Step 3: SFT data pipeline)

Plan v3, Step 3: download one month of real Lichess games -> sample positions by game phase -> label each with
Stockfish -> save to Drive as Parquet. This is the data SFT (Step 5) trains on.

**Gate:**
1. The labeled dataset is saved in Drive.
2. The phase mix matches `configs/data.yaml`.
3. The Stockfish CPU time spent labeling is logged (it counts against the SFT compute budget).

**Run it twice.** First with `TIMING_RUN = True` (5,000 positions) to measure labeling speed on Colab's CPU, then
with `TIMING_RUN = False` for the full dataset. Each run saves to its own folder.

**Safe to interrupt.** Labels are saved in chunks. If Colab disconnects, run the notebook again from the top:
finished chunks are kept and only the rest gets labeled. CPU only -- no GPU runtime needed.

In [ ]:
# Setup: clone the repo, install deps, mount Drive, fetch the pinned Stockfish.
import os
REPO_URL = "https://github.com/derekhe99/Chess_Bot"
BRANCH = "step-3-data"  # set to "main" once this branch is merged
REPO_DIR = "/content/Chess_Bot"
if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt

from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/chess-ai'
RAW = os.path.join(DRIVE_ROOT, 'data', 'raw')              # the downloaded Lichess month
PROCESSED = os.path.join(DRIVE_ROOT, 'data', 'processed')  # labeled datasets
os.makedirs(RAW, exist_ok=True)
os.makedirs(PROCESSED, exist_ok=True)

from engine.stockfish import ensure_stockfish, SF_VERSION
print("Stockfish:", SF_VERSION, ensure_stockfish())
CPU_MODEL = next((l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")), "unknown")
print("CPU:", CPU_MODEL, "| cores:", os.cpu_count())

## Parameters

Everything comes from `configs/data.yaml` except `TIMING_RUN`. Each dataset goes to its own Drive folder, named
after the month, label depth, and size, so the timing run and the full run never mix.

In [ ]:
import json
import time

import chess
import yaml

from data.annotate import annotate, load_labels
from data.dataset import load_datasets, split_by_game
from data.download import ensure_month, iter_games
from data.sample_positions import PHASES, PhaseRule, phase_quotas, position_key, sample_positions

TIMING_RUN = True   # True: 5,000 positions to measure speed. False: the full num_positions from the config.

cfg = yaml.safe_load(open("configs/data.yaml"))
MONTH = cfg["source_month"]
NUM_POSITIONS = 5000 if TIMING_RUN else cfg["num_positions"]
PHASE_MIX = cfg["phase_mix"]
RULE = PhaseRule(**cfg["phase_rule"])
DEPTH = cfg["stockfish_label_depth"]
WORKERS = os.cpu_count()   # one single-threaded Stockfish per core
OUT_DIR = os.path.join(PROCESSED, f"sft_{MONTH}_d{DEPTH}_n{NUM_POSITIONS}")

print(f"month {MONTH} | {NUM_POSITIONS} positions, targets {phase_quotas(NUM_POSITIONS, PHASE_MIX)}")
print(f"phase rule {RULE} | label depth {DEPTH} | {WORKERS} workers")
print("output:", OUT_DIR)

## Download
One month of rated games, cached in Drive `data/raw/` and checked against Lichess's published checksum. Only the
first run downloads; later runs reuse the file.

In [ ]:
path = ensure_month(MONTH, RAW)
print(f"{path.name}: {path.stat().st_size / 1e6:.0f} MB, checksum matches Lichess's published one")

## Sample positions
Reads games in file order and keeps at most one position per phase per game, skipping positions already taken,
until each phase's quota is full. Same month + same seed = same sample every run.

In [ ]:
t0 = time.perf_counter()
records, stats = sample_positions(iter_games(path), NUM_POSITIONS, PHASE_MIX, RULE, seed=cfg["seed"])
print(f"read {stats['games_read']} games in {time.perf_counter() - t0:.0f}s; "
      f"skipped {stats['duplicates_skipped']} repeated positions")
print("targets:", stats["targets"])
print("counts: ", stats["counts"], "" if stats["complete"] else "  <- SHORT: the month ran out of games")
os.makedirs(OUT_DIR, exist_ok=True)
with open(os.path.join(OUT_DIR, "sampling_stats.json"), "w") as f:
    json.dump({**stats, "month": MONTH, "phase_rule": vars(RULE), "seed": cfg["seed"]}, f, indent=2)

## Label with Stockfish
Each position gets Stockfish's best move and a win probability for the side to move, at a fixed depth. The
Stockfish CPU time is recorded -- it's charged to the SFT budget. If this cell is interrupted, just rerun it.

In [ ]:
manifest = annotate(records, OUT_DIR, depth=DEPTH, workers=WORKERS, chunk_size=cfg["label_chunk_size"])
cpu = manifest["stockfish_cpu_seconds"]
print(f"\n{manifest['rows']} labeled positions | Stockfish CPU {cpu / 60:.1f} min ({cpu / manifest['rows']:.3f} s/position)")
if manifest["labeled_this_run"]:
    wall_per = manifest["wall_seconds_this_run"] / manifest["labeled_this_run"]
    full = cfg["num_positions"]
    print(f"this run: {manifest['wall_seconds_this_run'] / 60:.1f} min wall")
    print(f"projected full dataset ({full}): {full * cpu / manifest['rows'] / 3600:.1f} CPU hours, "
          f"~{full * wall_per / 3600:.1f} hours wall on this machine")

## Gate check

In [ ]:
labels = load_labels(OUT_DIR)
manifest = json.load(open(os.path.join(OUT_DIR, "manifest.json")))
targets = phase_quotas(NUM_POSITIONS, PHASE_MIX)
counts = {p: int((labels["phase"] == p).sum()) for p in PHASES}
legal = all(chess.Move.from_uci(m) in chess.Board(f).legal_moves for f, m in zip(labels["fen"], labels["best_move"]))
keys = labels["fen"].map(lambda f: position_key(chess.Board(f)))
train_df, val_df = split_by_game(labels, cfg["val_fraction"], cfg["seed"])

checks = {
    f"labeled dataset saved to Drive ({len(labels)} positions)":
        os.path.exists(os.path.join(OUT_DIR, "labels.parquet")) and len(labels) == NUM_POSITIONS,
    f"phase mix matches config {counts}": counts == targets,
    f"labeling CPU time logged ({manifest['stockfish_cpu_seconds'] / 60:.1f} min)": manifest["stockfish_cpu_seconds"] > 0,
    "every label move is legal; win probabilities in [0, 1]": legal and labels["win_prob"].between(0, 1).all(),
    "no repeated positions": keys.is_unique,
    f"train / validation share no game ({len(train_df)} / {len(val_df)} positions)":
        set(train_df["game_id"]).isdisjoint(set(val_df["game_id"])),
}
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", "-", name)
GATE = all(checks.values())
print("\nGATE", "PASSED" if GATE else "NOT PASSED -- look at the failing line")

## What training will see
Plain records -- the board as text, Stockfish's move, the win probability. The prompt around them is built in
Step 4 (`model/llm_policy.py`), so training and play read exactly the same text.

In [ ]:
train, val = load_datasets(os.path.join(OUT_DIR, "labels.parquet"), "structured", cfg["val_fraction"], cfg["seed"])
print(f"train {len(train)} | validation {len(val)}\n")
item = train[0]
print(item["board_text"])
print(f"\nlabel move: {item['label_move']} | win prob (side to move): {item['label_value']:.3f} | phase: {item['phase']}")
print("\nsame position as FEN:", item["fen"])